# Part 1 — Data Preparation

Notebook này khám phá và chuẩn bị dữ liệu email spam. Cuối phần, dữ liệu được làm sạch và chia thành tập Train/Test theo tỷ lệ 80/20. **Chưa có bước huấn luyện mô hình trong notebook này.**

## 1. Mục tiêu

- Đọc dữ liệu email gốc và kiểm tra cấu trúc.
- Kiểm tra giá trị thiếu, phân bố nhãn và nội dung trùng.
- Chuẩn hóa nội dung email bằng hàm `clean_text()` dùng chung trong `src/preprocess.py`.
- Chia dữ liệu có phân tầng theo nhãn để giữ tỷ lệ spam/không spam gần tương đương ở hai tập.
- Lưu `train.csv` và `test.csv` vào `data/processed/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Tìm thư mục gốc để notebook chạy được dù kernel khởi động từ root hoặc notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "preprocess.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy src/preprocess.py từ thư mục hiện tại.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import (
    clean_text,
    prepare_dataset,
    save_prepared_datasets,
    split_dataset,
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spam_ham_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA_PATH}")

Project root: E:\ML_Classification
Raw data: E:\ML_Classification\data\raw\spam_ham_dataset.csv


## 2. Đọc dữ liệu gốc

Mỗi hàng là một email. `text` là nội dung đầu vào; `label_num` là nhãn cần dự đoán (`0` không spam, `1` spam). Các cột phụ của CSV gốc sẽ không được dùng làm đặc trưng trong tập dữ liệu đã chuẩn bị.

In [2]:
if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu: {RAW_DATA_PATH}")

df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Kích thước ban đầu: {df_raw.shape[0]:,} hàng × {df_raw.shape[1]} cột")
print("Các cột:", df_raw.columns.tolist())
sample_view = df_raw.head().copy()
sample_view["text"] = sample_view["text"].astype(str).str.slice(0, 220)
display(sample_view)

Kích thước ban đầu: 5,171 hàng × 4 cột
Các cột: ['Unnamed: 0', 'label', 'text', 'label_num']
   Unnamed: 0 label                                                                                                                                                                                                                                  text  label_num
0         605   ham    Subject: enron methanol ; meter # : 988291\r\nthis is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary\r\nflow data provided by daren } .\r\nplease override pop ' s daily volume { presently zero } to ref          0
1        2349   ham                                                                                                                                 Subject: hpl nom for january 9 , 2001\r\n( see attached file : hplnol 09 . xls )\r\n- hplnol 09 . xls          0
2        3624   ham      Subject: neon retreat\r\nho ho ho , we ' re around to that most wonderful time of the year - - - ne

## 3. Kiểm tra chất lượng dữ liệu

Trước khi làm sạch, xem số giá trị thiếu theo cột, số lượng từng nhãn và số nội dung email trùng nguyên bản. Các kiểm tra này giúp phát hiện dữ liệu cần xử lý trước khi chia tập.

In [3]:
required_columns = {"text", "label_num"}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

print("Giá trị thiếu theo cột:")
display(df_raw.isna().sum().rename("missing_count").to_frame())

print("Phân bố nhãn ban đầu (bao gồm giá trị thiếu nếu có):")
display(df_raw["label_num"].value_counts(dropna=False).rename_axis("label_num").to_frame("count"))

print("Số nội dung text trùng nguyên bản:", int(df_raw.duplicated(subset=["text"]).sum()))

Giá trị thiếu theo cột:
            missing_count
Unnamed: 0              0
label                   0
text                    0
label_num               0
Phân bố nhãn ban đầu (bao gồm giá trị thiếu nếu có):
           count
label_num       
0           3672
1           1499
Số nội dung text trùng nguyên bản: 178


## 4. Chọn cột, xử lý thiếu và làm sạch

Chỉ giữ nội dung và nhãn. Những hàng thiếu nội dung hoặc nhãn không thể dùng cho phân loại có giám sát nên được loại sau khi đã thống kê. Giữ thêm `raw_text` để tương thích với bước trích xuất đặc trưng hiện có; cột này lưu nội dung gốc trước khi chuẩn hóa. Sau đó loại bản ghi trùng và áp dụng `clean_text()` từ `src/`.

In [4]:
df, prep_diagnostics = prepare_dataset(df_raw)

print(f"Hàng bị loại do thiếu nội dung hoặc nhãn: {prep_diagnostics['missing_rows_removed']:,}")
print(f"Email trùng nguyên bản đã loại: {prep_diagnostics['raw_duplicates_removed']:,}")
print(f"Email rỗng sau làm sạch: {prep_diagnostics['empty_after_cleaning']:,}")
print(f"Nội dung trùng sau làm sạch: {prep_diagnostics['cleaned_duplicates']:,}")
print(f"Nội dung có nhãn mâu thuẫn: {prep_diagnostics['conflicting_labels']:,}")
print(f"Email trùng phát sinh sau làm sạch đã loại: {prep_diagnostics['cleaned_duplicates_removed']:,}")
print("Ví dụ trước làm sạch:\n", prep_diagnostics['sample_before'][:300])
print("\nVí dụ sau làm sạch:\n", prep_diagnostics['sample_after'][:300])
print(f"\nKích thước dữ liệu sau chuẩn bị: {df.shape[0]:,} hàng × {df.shape[1]} cột")
display(df["label_num"].value_counts().sort_index().rename_axis("label_num").to_frame("count"))

Hàng bị loại do thiếu nội dung hoặc nhãn: 0
Email trùng nguyên bản đã loại: 178
Email rỗng sau làm sạch: 0
Nội dung trùng sau làm sạch: 99
Nội dung có nhãn mâu thuẫn: 0
Email trùng phát sinh sau làm sạch đã loại: 99
Ví dụ trước làm sạch:
 Subject: enron methanol ; meter # : 988291
this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary
flow data provided by daren } .
please override pop ' s daily volume { presently zero } to reflect daily
activity you can obtain from gas control .
this change is needed as

Ví dụ sau làm sạch:
 subject: enron methanol ; meter # : 988291 this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary flow data provided by daren } . please override pop ' s daily volume { presently zero } to reflect daily activity you can obtain from gas control . this change is needed asap fo

Kích thước dữ liệu sau chuẩn bị: 4,894 hàng × 3 cột
           count
label_num       
0           3432
1           1462


## 5. Chia Train/Test

Dành 80% dữ liệu cho Train và 20% cho Test. `stratify` giữ tỷ lệ nhãn gần như nhau giữa hai tập; `random_state=42` giúp tái lập phép chia. Nội dung và nhãn cùng nằm trong mỗi CSV, để các hàng luôn được ghép đúng.

In [5]:
train_df, test_df = split_dataset(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

saved_paths = save_prepared_datasets(df, train_df, test_df, PROCESSED_DIR)
print(f"Đã lưu dữ liệu sạch: {saved_paths['clean']}")
print(f"Đã lưu tập Train:    {saved_paths['train']}")
print(f"Đã lưu tập Test:     {saved_paths['test']}")

Đã lưu dữ liệu sạch: E:\ML_Classification\data\processed\emails_clean.csv
Đã lưu tập Train:    E:\ML_Classification\data\processed\train.csv
Đã lưu tập Test:     E:\ML_Classification\data\processed\test.csv


## 6. Kiểm tra kết quả chia

Kiểm tra kích thước hai tập, tỷ lệ nhãn và giao nhau của nội dung. Tập Test sẽ được giữ riêng; các notebook huấn luyện sau chỉ dùng Test để đánh giá cuối cùng.

In [6]:
print(f"Train: {len(train_df):,} hàng ({len(train_df) / len(df):.1%})")
print(f"Test:  {len(test_df):,} hàng ({len(test_df) / len(df):.1%})")

print("\nTỷ lệ nhãn toàn bộ dữ liệu:")
display(df["label_num"].value_counts(normalize=True).sort_index().rename("proportion").to_frame())
print("Tỷ lệ nhãn Train:")
display(train_df["label_num"].value_counts(normalize=True).sort_index().rename("proportion").to_frame())
print("Tỷ lệ nhãn Test:")
display(test_df["label_num"].value_counts(normalize=True).sort_index().rename("proportion").to_frame())

shared_texts = set(train_df["text"]) & set(test_df["text"])
print("Số nội dung trùng giữa Train và Test:", len(shared_texts))
if shared_texts:
    raise AssertionError("Phát hiện nội dung xuất hiện ở cả Train và Test.")

assert len(train_df) + len(test_df) == len(df)
assert set(train_df["label_num"].unique()) == {0, 1}
assert set(test_df["label_num"].unique()) == {0, 1}
print("Kiểm tra chia dữ liệu hoàn tất. Notebook chưa huấn luyện mô hình.")

Train: 3,915 hàng (80.0%)
Test:  979 hàng (20.0%)

Tỷ lệ nhãn toàn bộ dữ liệu:
           proportion
label_num            
0            0.701267
1            0.298733
Tỷ lệ nhãn Train:
           proportion
label_num            
0            0.701149
1            0.298851
Tỷ lệ nhãn Test:
           proportion
label_num            
0            0.701736
1            0.298264
Số nội dung trùng giữa Train và Test: 0
Kiểm tra chia dữ liệu hoàn tất. Notebook chưa huấn luyện mô hình.
